In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/hsbc-bank-uks-largest-bank/hsbc.csv


In [3]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [4]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/hsbc-bank-uks-largest-bank/hsbc.csv"
df = pd.read_csv(file_path)

In [5]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)


In [6]:
# =====================================================
# 4. ATLAS HEADER
# =====================================================
from IPython.display import display, HTML

def atlas_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #1e3c72, #2a5298);
        padding: 20px; border-radius: 14px; margin-top:20px;">
        <h1 style="color:#dbeafe; text-align:center;">{text}</h1>
    </div>
    """))

atlas_header("📊 Global Price Structure")

In [7]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='HSBC Price Trend')
fig.show()

In [8]:
# =====================================================
# 6. RETURNS & CUMULATIVE PERFORMANCE
# =====================================================
atlas_header("📈 Return Engine")

df['returns'] = df['close'].pct_change()
df['cum_returns'] = (1 + df['returns']).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['cum_returns'], name='Cumulative Returns'))
fig.show()

In [9]:
# =====================================================
# 7. DRAWDOWN ANALYSIS
# =====================================================
atlas_header("📉 Drawdown Control")

rolling_max = df['cum_returns'].cummax()
df['drawdown'] = df['cum_returns'] / rolling_max - 1

fig = px.area(df, y='drawdown', title='Drawdown Curve')
fig.show()

In [10]:
# =====================================================
# 8. RISK METRICS DASHBOARD
# =====================================================
atlas_header("⚖️ Risk Metrics")

vol = df['returns'].std() * np.sqrt(252)
sharpe = df['returns'].mean() / df['returns'].std() * np.sqrt(252)
max_dd = df['drawdown'].min()

print(f"Annual Volatility: {vol:.4f}")
print(f"Sharpe Ratio: {sharpe:.4f}")
print(f"Max Drawdown: {max_dd:.4f}")


Annual Volatility: 0.2701
Sharpe Ratio: 0.3731
Max Drawdown: -0.7447


In [11]:
# =====================================================
# 9. RISK PARITY LOGIC (SINGLE ASSET PROXY)
# =====================================================
atlas_header("🧠 Risk Parity Thinking")

# Position sizing based on inverse volatility
df['rolling_vol'] = df['returns'].rolling(20).std()
df['weight'] = 1 / df['rolling_vol']
df['weight'] = df['weight'] / df['weight'].sum()

fig = px.line(df, y='weight', title='Dynamic Risk Weight')
fig.show()

In [12]:
# =====================================================
# 10. VOLATILITY TARGETING STRATEGY
# =====================================================
atlas_header("🎯 Volatility Targeting")

target_vol = 0.15
df['scaled_returns'] = df['returns'] * (target_vol / df['rolling_vol'])
df['strategy'] = (1 + df['scaled_returns']).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['cum_returns'], name='Buy & Hold'))
fig.add_trace(go.Scatter(x=df.index, y=df['strategy'], name='Vol Targeted'))
fig.show()

In [13]:
# =====================================================
# 11. STRESS ZONE DETECTION
# =====================================================
atlas_header("🔥 Stress Zones")

z = (df['returns'] - df['returns'].mean()) / df['returns'].std()
df['stress'] = np.abs(z) > 2

fig = px.scatter(df, x=df.index, y='returns', color='stress')
fig.show()

In [14]:
# =====================================================
# 12. FINAL INSIGHTS
# =====================================================
atlas_header("📌 Portfolio Intelligence")

print("""
1. Drawdowns define real risk, not volatility alone.
2. Risk parity improves capital allocation stability.
3. Volatility targeting smooths equity curve.
4. Stress zones highlight tail risk events.
5. Portfolio thinking > single trade thinking.
""")

# =====================================================
# END
# =====================================================



1. Drawdowns define real risk, not volatility alone.
2. Risk parity improves capital allocation stability.
3. Volatility targeting smooths equity curve.
4. Stress zones highlight tail risk events.
5. Portfolio thinking > single trade thinking.

